[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch03-ml-workflow/03-vong-doi-thu-nho.ipynb)

# Một vòng đời thu nhỏ: từ ngưỡng lâm sàng tới suy giảm âm thầm

Chương 3 đi qua sáu bước của vòng đời hệ thống học máy (ML lifecycle) bằng một hệ thống sàng lọc bệnh
võng mạc do đái tháo đường cho các phòng khám vùng xa. Sổ tay này chạy cả sáu bước, thu nhỏ, trên
**ảnh võng mạc giả lập**: định nghĩa bài toán thành hai cổng kiểm tra, thu thập dữ liệu từ hai loại máy
chụp, huấn luyện, đánh giá và kiểm định, triển khai cho sáu phòng khám, rồi theo dõi (monitoring) khi
một phòng khám thay máy chụp. Ảnh là giả lập nên các con số không mang nghĩa lâm sàng. Thứ đáng xem là
hình dạng của từng thất bại, và tín hiệu nào báo nó trước.

**Bạn sẽ làm:**
1. viết yêu cầu của bước định nghĩa bài toán thành hai cổng: độ nhạy (sensitivity) và độ đặc hiệu
   (specificity);
2. sinh ảnh từ ba loại máy chụp và huấn luyện một mô hình trên ảnh của phòng lab;
3. đánh giá trên phòng lab, rồi kiểm định ở nơi dùng máy chụp cũ: đường ROC, hiệu chuẩn (calibration),
   đánh giá tách theo nhóm;
4. thử thêm dữ liệu lab, rồi sửa bằng tiền xử lý và dữ liệu thực địa, và cho mô hình qua cổng phát
   hành;
5. triển khai 14 tuần cho sáu phòng khám, để một phòng khám đổi sang máy chụp mới, rồi xem tín hiệu
   theo dõi nào báo trước;
6. thử quay lại phiên bản trước (rollback), rồi huấn luyện lại bằng dữ liệu từ vòng phản hồi.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import pandas as pd
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a5cc2"]  # thứ tự màu cố định, như các sổ tay khác
print("sẵn sàng")

## 1. Định nghĩa bài toán: hai cổng

Chương không dùng độ chính xác cho bài toán này. Yêu cầu minh hoạ của nó là độ nhạy trên 90 %, để hạn
chế bỏ sót bệnh cần chuyển tuyến, và độ đặc hiệu trên 80 %, để tuyến trên không bị ngập trong ca dương
tính giả. Chương cũng nói ngưỡng thật phụ thuộc mục đích sử dụng, quần thể bệnh nhân và bằng chứng pháp
lý. Ở đây, hai yêu cầu đó thành một hàm trả lời có hoặc không.

In [ ]:
NHAY_MIN = sach(90, nguon="> 90 % độ nhạy · > 80 % độ đặc hiệu")
DAC_HIEU_MIN = sach(80, nguon="> 90 % độ nhạy · > 80 % độ đặc hiệu")


def do_nhay_dac_hieu(diem, y, nguong):
    du_doan = diem >= nguong
    return 100 * du_doan[y == 1].mean(), 100 * (~du_doan[y == 0]).mean()


def qua_cong(nhay, dac_hieu):
    return nhay > NHAY_MIN and dac_hieu > DAC_HIEU_MIN


print(f"cổng: độ nhạy > {NHAY_MIN} % và độ đặc hiệu > {DAC_HIEU_MIN} %")
print("hệ thống đã công bố mà chương dẫn (Gulshan và cộng sự, 2016):",
      f"{sach(128_000, nguon='128.000 ảnh · AUC 0,99'):,} ảnh,",
      f"AUC {sach(0.99, nguon='128.000 ảnh · AUC 0,99')},",
      f"độ nhạy {sach(97.5, nguon='độ nhạy 97,5 %')} %,",
      f"độ đặc hiệu {sach(93.4, nguon='độ đặc hiệu 93,4 %')} %")

## 2. Ảnh võng mạc giả lập, từ ba loại máy chụp

Mỗi ảnh là một đĩa võng mạc 32×32 điểm ảnh, 3 kênh màu, có đĩa thị và vài mạch máu. Ảnh có bệnh mang
thêm vài chấm đỏ sẫm nhỏ, kiểu vi phình mạch. Ba loại máy chụp khác nhau ở độ sáng, độ mờ, độ nhiễu, độ
lệch tâm và hồ sơ màu. Mọi tham số dưới đây là **giả định** của sổ tay, kể cả tỉ lệ có bệnh 30 %, cao
hơn thực tế để mỗi tuần có đủ ca có bệnh mà đo độ nhạy.

* **Máy A** là máy của phòng lab nơi dữ liệu phát triển được thu.
* **Máy B** là máy đời cũ ở một số phòng khám: ảnh tối hơn, ngả đỏ vì kênh lục và kênh lam yếu, hơi mờ
  và lệch tâm hơn.
* **Máy C** là máy đời mới mà một phòng khám sẽ mua giữa chừng. Ảnh của nó sắc nét như máy A, nhưng
  có hồ sơ màu khác: đường cong độ sáng (gamma) 0.65, kênh lam mạnh hơn 15 %.

In [ ]:
S = 32
YY, XX = np.mgrid[0:S, 0:S].astype(np.float32)
MAY = {
    "A": dict(sang=(0.9, 1.1), mo=(0.0, 0.3), nhieu=0.02, lech=0.8),
    "B": dict(sang=(0.5, 0.7), mo=(0.1, 0.4), nhieu=0.01, lech=1.5, gamma=1.0, can_bang=(1.0, 0.6, 0.5)),
    "C": dict(sang=(0.9, 1.1), mo=(0.0, 0.3), nhieu=0.02, lech=0.8, gamma=0.65, can_bang=(0.95, 1.0, 1.15)),
}
TI_LE_BENH = 0.3  # giả định


def _dom(cx, cy, s):
    """Các đốm Gauss: cx, cy, s có dạng (n, k), trả về (n, k, S, S)."""
    return np.exp(-((XX - cx[..., None, None]) ** 2 + (YY - cy[..., None, None]) ** 2) / (2 * s[..., None, None] ** 2))


def lam_mo(anh, sigma):
    """Làm mờ từng ảnh với độ mờ riêng của nó (lọc Gauss tách theo hai chiều)."""
    ra = anh.clone()
    sr = np.round(sigma, 1)
    for s in np.unique(sr):
        if s < 0.15:
            continue
        chi_so = np.where(sr == s)[0]
        r = int(math.ceil(3 * s))
        t = torch.arange(-r, r + 1, dtype=torch.float32)
        k = torch.exp(-t ** 2 / (2 * s * s))
        k = k / k.sum()
        x = anh[chi_so].reshape(-1, 1, S, S)
        x = F.conv2d(F.pad(x, (r, r, 0, 0), mode="replicate"), k.view(1, 1, 1, -1))
        x = F.conv2d(F.pad(x, (0, 0, r, r), mode="replicate"), k.view(1, 1, -1, 1))
        ra[chi_so] = x.reshape(len(chi_so), 3, S, S)
    return ra


def chup(n, may, rng, ti_le_benh=TI_LE_BENH):
    """n ảnh giả lập từ một loại máy chụp: trả về ảnh (n, 3, S, S) trong [0, 1] và nhãn có bệnh."""
    y = rng.random(n) < ti_le_benh
    cx, cy = (15.5 + rng.normal(0, may["lech"], (2, n))).astype(np.float32)
    r = 13 + rng.uniform(-0.5, 0.5, n)
    d = np.hypot(XX - cx[:, None, None], YY - cy[:, None, None])
    dia = 1 / (1 + np.exp(-(r[:, None, None] - d) / 0.7))
    mau = np.array([0.80, 0.38, 0.18]) * (1 + rng.normal(0, 0.05, (n, 3)))
    anh = mau[:, :, None, None] * (1 - 0.35 * np.clip(d / r[:, None, None], 0, 1.2) ** 2)[:, None]
    huong = np.where(rng.random(n) < 0.5, 1, -1)
    ox, oy = cx + huong * 6 + rng.normal(0, 0.7, n), cy + rng.normal(0, 1.5, n)  # đĩa thị
    anh = anh + np.array([0.15, 0.30, 0.20])[None, :, None, None] * _dom(ox[:, None], oy[:, None], np.full((n, 1), 2.2))
    # mạch máu: 4 đoạn đi ra từ mép đĩa thị
    g = rng.uniform(0, 2 * np.pi, (n, 4))
    L = rng.uniform(6, 12, (n, 4))
    x0, y0 = ox[:, None] + 2.5 * np.cos(g), oy[:, None] + 2.5 * np.sin(g)
    vx, vy = L * np.cos(g), L * np.sin(g)
    px, py = XX - x0[..., None, None], YY - y0[..., None, None]
    tt = np.clip((px * vx[..., None, None] + py * vy[..., None, None]) / (L ** 2)[..., None, None], 0, 1)
    dd = np.hypot(px - tt * vx[..., None, None], py - tt * vy[..., None, None])
    v = np.exp(-dd ** 2 / (2 * 0.5 ** 2)) * (1 - 0.6 * tt)
    anh = anh * np.prod(1 - np.array([0.08, 0.25, 0.25])[None, None, :, None, None] * v[:, :, None], axis=1)
    # tổn thương: 1 + Poisson(1.2) chấm đỏ sẫm, chỉ ở ảnh có bệnh
    K = 6
    so = np.where(y, np.minimum(1 + rng.poisson(1.2, n), K), 0)
    rr, gg = rng.uniform(0, 1, (n, K)) * (r[:, None] - 3), rng.uniform(0, 2 * np.pi, (n, K))
    a = rng.uniform(0.5, 0.8, (n, K)) * (np.arange(K)[None] < so[:, None])
    b = a[..., None, None] * _dom(cx[:, None] + rr * np.cos(gg), cy[:, None] + rr * np.sin(gg), rng.uniform(0.8, 1.3, (n, K)))
    anh = anh * np.prod(1 - np.array([0.25, 1.0, 1.0])[None, None, :, None, None] * b[:, :, None], axis=1)
    anh = anh * dia[:, None] * rng.uniform(*may["sang"], n)[:, None, None, None]
    if "can_bang" in may:  # hồ sơ màu của máy
        anh = np.clip(anh * np.array(may["can_bang"])[None, :, None, None], 0, 1) ** may["gamma"]
    t = lam_mo(torch.from_numpy(anh.astype(np.float32)), rng.uniform(*may["mo"], n))
    t = t + torch.from_numpy(rng.normal(0, may["nhieu"], t.shape).astype(np.float32))
    return t.clamp(0, 1), y.astype(np.int64)


def chup_nhieu(n, may, rng, lo=2000):
    """Như chup, theo từng lô, để bộ nhớ tạm không phình theo n."""
    phan = [chup(min(lo, n - i), may, rng) for i in range(0, n, lo)]
    return torch.cat([p[0] for p in phan]), np.concatenate([p[1] for p in phan])


rng = np.random.default_rng(SEED)
X_lab, y_lab = chup_nhieu(5000, MAY["A"], rng)
X_b, y_b = chup_nhieu(2000, MAY["B"], rng)
X_c_mau, y_c_mau = chup(8, MAY["C"], np.random.default_rng(SEED + 1))
print(f"phòng lab, máy A: {len(y_lab):,} ảnh, {100 * y_lab.mean():.1f} % có bệnh")
print(f"thử nghiệm thực địa, máy B: {len(y_b):,} ảnh, {100 * y_b.mean():.1f} % có bệnh")

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(8, 6.4))
nguon_mau = [("A", X_lab, y_lab), ("B", X_b, y_b), ("C", X_c_mau, y_c_mau)]
for hang, (ten, X, y) in zip(axes, nguon_mau):
    chon = list(np.where(y == 0)[0][:2]) + list(np.where(y == 1)[0][:2])
    for ax, i in zip(hang, chon):
        ax.imshow(X[i].permute(1, 2, 0).numpy())
        ax.set_title(f"máy {ten}, {'có bệnh' if y[i] else 'không bệnh'}", fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
fig.suptitle("Ảnh võng mạc giả lập: chấm đỏ sẫm nhỏ là tổn thương", fontsize=10)
plt.tight_layout()
plt.show()

## 3. Mô hình đầu tiên, trên ảnh của phòng lab

Mô hình ở đây cố ý nhỏ, để cả vòng đời chạy trong vài giây. Một bộ lọc hiệu hai Gauss làm nổi các
chấm tối hơn vùng quanh nó. Từ đó ta lấy vài đặc trưng cho mỗi kênh: phản hồi mạnh nhất, trung bình
năm phản hồi mạnh nhất, phản hồi trung bình, và độ sáng trung bình của vùng đĩa. Một hồi quy logistic
biến các đặc trưng thành điểm từ 0 tới 1.

Dữ liệu lab chia ba phần: 3,000 ảnh huấn luyện, 1,000 ảnh kiểm định (validation) để chọn ngưỡng, 1,000
ảnh kiểm tra. Ngưỡng được chọn sao cho độ nhạy trên tập kiểm định đạt 96 %, cao hơn cổng 90 % một
khoảng dự phòng, vì độ nhạy đo trên tập kiểm tra sẽ lệch khỏi tập kiểm định vài điểm. Con số 96 % là
**giả định** của sổ tay.

**Dự đoán:** mô hình này qua cổng trên ảnh lab. Nó có qua ở phòng khám dùng máy B không?

In [ ]:
MUC_NHAY = 96  # giả định: độ nhạy mục tiêu khi chọn ngưỡng


def chuan_hoa_kenh(X):
    """Tiền xử lý: chia mỗi kênh màu cho độ sáng trung bình của chính kênh đó trong vùng đĩa."""
    R = X[:, 0]
    vung = (R > 0.3 * R.amax((1, 2), keepdim=True)).float()[:, None]
    tb = (X * vung).sum((2, 3)) / vung.sum((2, 3))
    return (X / tb[:, :, None, None] * 0.4 * vung).clamp(0, 3)


def _ma_tran_loc(s):
    """Lọc Gauss một chiều, kể cả lề lặp lại, viết thành một ma trận S×S."""
    r = int(math.ceil(3 * s))
    t = torch.arange(-r, r + 1, dtype=torch.float32)
    k = torch.exp(-t ** 2 / (2 * s * s))
    k = k / k.sum()
    M = torch.zeros(S, S)
    for i in range(S):
        for j, w in zip(range(i - r, i + r + 1), k):
            M[i, min(max(j, 0), S - 1)] += w
    return M


_LOC = {s: _ma_tran_loc(s) for s in (0.8, 1.2, 2.0, 3.0)}


def _loc(X, s):
    """Làm mờ Gauss theo cả hai chiều: nhân ma trận ở hai phía, nhanh hơn một phép tích chập 2D."""
    return _LOC[s] @ X @ _LOC[s].T


def dac_trung(X, tien_xu_ly=False):
    if tien_xu_ly:
        X = chuan_hoa_kenh(X)
    R = X[:, 0:1]
    vung = (R > 0.3 * R.amax((2, 3), keepdim=True)).float()
    vung = -F.max_pool2d(-vung, 5, 1, 2)  # co vùng lại để bỏ mép đĩa
    ra = []
    for s1, s2 in [(0.8, 2.0), (1.2, 3.0)]:
        hieu = ((_loc(X, s2) - _loc(X, s1)) * vung).flatten(2)  # dương ở chỗ tối hơn xung quanh
        top = hieu.topk(5, dim=2).values
        ra += [top[:, :, 0], top.mean(2), hieu.mean(2)]
    ra.append((X * vung).sum((2, 3)) / vung.sum((2, 3)))
    return torch.cat(ra, 1).numpy()


class MoHinh:
    def __init__(self, ten, X, y, tien_xu_ly=False):
        self.ten, self.tien_xu_ly = ten, tien_xu_ly
        self.lr = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000))
        self.lr.fit(dac_trung(X, tien_xu_ly), y)

    def diem(self, X):
        return self.lr.predict_proba(dac_trung(X, self.tien_xu_ly))[:, 1]

    def chon_nguong(self, tap_kiem_dinh):
        """Ngưỡng cao nhất sao cho độ nhạy ở MỌI nơi trong tập kiểm định đạt MUC_NHAY."""
        ds = []
        for X, y in tap_kiem_dinh:
            duong = np.sort(self.diem(X)[y == 1])
            ds.append(duong[int(np.floor((1 - MUC_NHAY / 100) * len(duong)))])
        self.nguong = float(min(ds))
        return self


LAB_HL, LAB_KD, LAB_KT = (X_lab[:3000], y_lab[:3000]), (X_lab[3000:4000], y_lab[3000:4000]), (X_lab[4000:], y_lab[4000:])
B_HL, B_KD, B_KT = (X_b[:400], y_b[:400]), (X_b[400:1000], y_b[400:1000]), (X_b[1000:], y_b[1000:])


def bang_cong(mo_hinh_list, cac_noi):
    hang = []
    for m in mo_hinh_list:
        for ten_noi, (X, y) in cac_noi.items():
            s = m.diem(X)
            nhay, dac_hieu = do_nhay_dac_hieu(s, y, m.nguong)
            hang.append({"mô hình": m.ten, "nơi": ten_noi, "AUC": roc_auc_score(y, s), "độ nhạy (%)": nhay,
                         "độ đặc hiệu (%)": dac_hieu, "chuyển tuyến (%)": 100 * np.mean(s >= m.nguong),
                         "qua cổng": "có" if qua_cong(nhay, dac_hieu) else "KHÔNG"})
    return pd.DataFrame(hang).round(3)


t0 = time.perf_counter()
v0 = MoHinh("v0: chỉ ảnh lab", *LAB_HL).chon_nguong([LAB_KD])
do_tren_may("huấn luyện v0 trên 3,000 ảnh, kể cả tính đặc trưng", time.perf_counter() - t0, "s")
bang_v0 = bang_cong([v0], {"lab, máy A": LAB_KT, "phòng khám, máy B": B_KT})
bang_v0

Chương kể một cảnh tương tự: mô hình đạt AUC 0.99 trên dữ liệu nghiên cứu, rồi được thử ở một phòng
khám nông thôn tại Chiang Mai, nơi một kỹ thuật viên mới được đào tạo vận hành một máy chụp đáy mắt
năm năm tuổi. Ở đó độ nhạy rơi xuống 78 %. Không có dòng code nào hỏng. Mô hình chỉ chưa bao giờ thấy
ảnh mờ, thiếu sáng và lệch khung như vậy.

Hai đồ thị dưới đây đọc bảng trên kỹ hơn. Bên trái là đường ROC ở hai nơi, với vùng mà cả hai cổng
đều qua: tỉ lệ dương tính giả (false positive rate) dưới 20 % và độ nhạy trên 90 %. Bên phải là đồ thị
hiệu chuẩn: trong số ảnh được chấm khoảng 0.7, có đúng khoảng 70 % là có bệnh không?

In [ ]:
print(f"chương: độ nhạy thực địa {sach(78, trich='Sensitivity drops to 78 percent')} % (ở cảnh của chương, "
      f"không phải con số của mô phỏng này)")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.6))
ax1.add_patch(plt.Rectangle((0, NHAY_MIN / 100), 1 - DAC_HIEU_MIN / 100, 1 - NHAY_MIN / 100,
                            color=MAU[2], alpha=0.15, lw=0, label="vùng qua cả hai cổng"))
for (ten, (X, y)), mau in zip({"lab, máy A": LAB_KT, "phòng khám, máy B": B_KT}.items(), MAU):
    s = v0.diem(X)
    fpr, tpr, _ = roc_curve(y, s)
    nhay, dac_hieu = do_nhay_dac_hieu(s, y, v0.nguong)
    ax1.plot(fpr, tpr, color=mau, lw=2, label=f"{ten}, AUC {roc_auc_score(y, s):.3f}")
    ax1.plot([1 - dac_hieu / 100], [nhay / 100], "o", color=mau, ms=8, mec="white", mew=1.5)
    bien = np.linspace(0, 1, 11)
    ngan = np.clip(np.digitize(s, bien) - 1, 0, 9)
    co = [k for k in range(10) if (ngan == k).sum() >= 20]
    ax2.plot([s[ngan == k].mean() for k in co], [y[ngan == k].mean() for k in co], "o-", color=mau, lw=2, ms=5,
             label=ten)
ax1.text(0.21, 0.5, "chấm tròn: điểm vận hành,\nngưỡng chọn trên tập kiểm định lab", fontsize=8, color="0.3")
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1.01)
ax1.set_xlabel("tỉ lệ dương tính giả = 1 − độ đặc hiệu")
ax1.set_ylabel("độ nhạy")
ax1.set_title("ROC của v0 ở hai nơi (ảnh giả lập)")
ax1.legend(loc="lower right", fontsize=8)
ax2.plot([0, 1], [0, 1], color="0.6", lw=1, ls="--", label="hiệu chuẩn hoàn hảo")
ax2.set_xlabel("điểm trung bình của mô hình trong mỗi ngăn")
ax2.set_ylabel("tỉ lệ có bệnh thật trong ngăn")
ax2.set_title("Hiệu chuẩn của v0 (ảnh giả lập)")
ax2.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Đọc hai đồ thị cùng nhau. Nếu đường ROC của máy B vẫn đi qua vùng xanh mà chấm tròn của nó nằm ngoài,
mô hình vẫn còn phân biệt được ảnh có bệnh ở máy B, chỉ là ngưỡng chọn ở lab đặt sai chỗ, và đồ thị
hiệu chuẩn cho thấy điểm số của máy B đã lệch. Một manh mối nằm ở chính các đặc trưng: trong đó có độ
sáng trung bình của từng kênh, và ở lab, tổn thương làm kênh lục và kênh lam tối đi một chút. Ở máy B,
hai kênh đó yếu trên mọi ảnh.

Ở cảnh của chương, ảnh lạ làm mô hình bỏ sót bệnh. Ở đây, xem dòng của máy B trong bảng để biết cổng
nào trượt: ảnh lạ có thể đẩy điểm số theo chiều nào cũng được, thành bỏ sót bệnh hoặc thành chuyển
tuyến tràn lan.

Chương gọi cách nhìn này là đánh giá tách theo nhóm: một mô hình đạt 94 % tổng thể có thể xuống dưới
80 % ở một nhóm ảnh. Gộp hai nơi lại, con số tổng trộn một nơi tốt với một nơi tệ, nên nó không nói vấn
đề nằm ở đâu. Khi nơi tệ chỉ chiếm một phần nhỏ dữ liệu, con số tổng còn có thể giấu hẳn vấn đề. Phần 5
sẽ gặp lại chuyện này.

In [ ]:
X_gop, y_gop = torch.cat([LAB_KT[0], B_KT[0]]), np.concatenate([LAB_KT[1], B_KT[1]])
nhay, dac_hieu = do_nhay_dac_hieu(v0.diem(X_gop), y_gop, v0.nguong)
dung = 100 * np.mean((v0.diem(X_gop) >= v0.nguong) == y_gop)
do_tren_may("v0, gộp lab và máy B: độ chính xác", dung, "%")
do_tren_may("v0, gộp lab và máy B: độ nhạy", nhay, "%")
do_tren_may("v0, gộp lab và máy B: độ đặc hiệu", dac_hieu, "%")
print("theo từng nơi (bảng ở trên):", ", ".join(f"{r['nơi']}: qua cổng {r['qua cổng']}" for _, r in bang_v0.iterrows()))
print(f"ví dụ của chương: {sach(94, nguon='94 % → dưới 80 %')} % tổng thể, "
      f"dưới {sach(80, nguon='94 % → dưới 80 %')} % ở một nhóm")

## 4. Thêm dữ liệu lab, hay đổi cách chuẩn bị dữ liệu

Phản xạ đầu tiên là thu thêm dữ liệu. Chương gọi đó là một cạm bẫy: mở rộng việc thu thập dữ liệu
trước khi kiểm giá trị biên của nó. Ô dưới đây huấn luyện lại v0 trên 250 tới 3,000 ảnh lab, và đo cả
hai nơi.

**Dự đoán:** với gấp 12 lần dữ liệu lab, độ đặc hiệu ở máy B tăng bao nhiêu?

In [ ]:
co_mau = [250, 500, 1000, 2000, 3000]
duong_hoc = []
for n in co_mau:
    m = MoHinh(f"lab {n}", LAB_HL[0][:n], LAB_HL[1][:n]).chon_nguong([LAB_KD])
    for ten, (X, y) in {"lab, máy A": LAB_KT, "phòng khám, máy B": B_KT}.items():
        duong_hoc.append((n, ten, *do_nhay_dac_hieu(m.diem(X), y, m.nguong)))
duong_hoc = pd.DataFrame(duong_hoc, columns=["ảnh lab", "nơi", "độ nhạy (%)", "độ đặc hiệu (%)"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for ax, cot, cong in [(ax1, "độ nhạy (%)", NHAY_MIN), (ax2, "độ đặc hiệu (%)", DAC_HIEU_MIN)]:
    for (ten, nhom), mau in zip(duong_hoc.groupby("nơi", sort=False), MAU):
        ax.plot(nhom["ảnh lab"], nhom[cot], "o-", color=mau, lw=2, label=ten)
    ax.axhline(cong, color="0.3", ls="--", lw=1,
               label=f"cổng của chương: độ nhạy > {NHAY_MIN} %, độ đặc hiệu > {DAC_HIEU_MIN} %")
    ax.set_xscale("log")
    ax.set_xticks(co_mau, [f"{n:,}" for n in co_mau])
    ax.minorticks_off()
    ax.set_xlabel("số ảnh lab dùng để huấn luyện")
    ax.set_ylabel(cot)
    ax.set_ylim(min(40, duong_hoc[cot].min() - 5), 101)
ax1.set_title("Độ nhạy theo lượng dữ liệu lab (ảnh giả lập)")
ax2.set_title("Độ đặc hiệu theo lượng dữ liệu lab (ảnh giả lập)")
fig.legend(*ax1.get_legend_handles_labels(), loc="lower center", ncol=3, fontsize=9, frameon=False)
plt.tight_layout(rect=(0, 0.07, 1, 1))
plt.show()
duong_hoc.round(1)

Đường của lab gần như phẳng: hồi quy logistic ở đây chỉ có 22 tham số, nên vài trăm ảnh đã gần đủ.
Đường của máy B thì không tiến dần về cổng: dữ liệu lab thêm vào giống dữ liệu cũ, nên nó không cho mô
hình biết gì về một máy chụp ngả đỏ. Điểm số ở máy B là chuyện ngoại suy, nên nó có thể nhảy từ trượt
cổng này sang trượt cổng kia chỉ vì đổi cỡ dữ liệu. Chương nói rõ cách sửa cho chính tình huống này: các
mô hình tối ưu cho ảnh chất lượng nghiên cứu có thể hỏng với ảnh do kỹ thuật viên ít kinh nghiệm chụp,
và cần một quy trình tiền xử lý chuẩn hoá chất lượng ảnh trước khi suy luận. Ta làm hai việc:

* tiền xử lý: chia mỗi kênh màu cho độ sáng trung bình của chính kênh đó, nên độ sáng và cân bằng
  trắng của máy không còn là manh mối;
* thu 1,000 ảnh có nhãn từ một đợt thử nghiệm ở phòng khám dùng máy B: 400 để huấn luyện, 600 để chọn
  ngưỡng. Ngưỡng giờ phải đạt độ nhạy 96 % ở **cả hai** nơi.

Việc thứ nhất có một tính chất kiểm được chắc chắn: nhân mỗi kênh với một hằng số, ví dụ làm ảnh tối
và ngả đỏ như máy B, thì ảnh sau tiền xử lý không đổi.

In [ ]:
anh_nga_do = X_lab[:200] * 0.6 * torch.tensor(MAY["B"]["can_bang"])[None, :, None, None]
assert torch.allclose(chuan_hoa_kenh(anh_nga_do), chuan_hoa_kenh(X_lab[:200]), atol=1e-5)
print("ảnh tối đi 0.6 lần và ngả đỏ như máy B: sau tiền xử lý giống hệt ảnh gốc, nên đặc trưng cũng vậy")

v0_tx = MoHinh("v0 + tiền xử lý", *LAB_HL, tien_xu_ly=True).chon_nguong([LAB_KD])
v1 = MoHinh("v1: tiền xử lý + ảnh máy B", torch.cat([LAB_HL[0], B_HL[0]]), np.concatenate([LAB_HL[1], B_HL[1]]),
            tien_xu_ly=True).chon_nguong([LAB_KD, B_KD])
bang_v1 = bang_cong([v0, v0_tx, v1], {"lab, máy A": LAB_KT, "phòng khám, máy B": B_KT})
bang_v1

Bảng trên là cổng phát hành của sổ tay: mỗi phiên bản, mỗi nơi, đo trên tập kiểm tra chưa dùng để huấn
luyện hay chọn ngưỡng. Chỉ phiên bản qua cổng ở mọi nơi mới được triển khai.

Có thể bảng của bạn cho thấy cả v0 + tiền xử lý lẫn v1 đều qua. Sổ tay vẫn chọn v1, vì ngưỡng của nó
được chọn trên tập kiểm định của cả hai loại máy. Ngưỡng của v0 + tiền xử lý chỉ được chọn trên ảnh lab,
nên nếu nó qua ở máy B thì đó là ngoại suy gặp may, chưa phải một điều đã kiểm trước khi đo. Nếu ở bảng
của bạn v1 trượt một cổng, thì theo đúng luật cổng, chưa phiên bản nào được triển khai: phần dưới vẫn
chạy v1 để bạn xem hệ thống theo dõi làm việc ra sao.

## 5. Triển khai, và một phòng khám đổi máy chụp

v1 được triển khai cho sáu phòng khám: PK1 tới PK3 dùng máy A, PK4 tới PK6 dùng máy B. Mỗi phòng khám
khám 50 bệnh nhân một ngày, như quy mô triển khai của chương. Ở ngày 28, PK2 thay máy A bằng máy C. Đây
chính là cảnh của chương: một phòng khám nâng cấp máy chụp, máy mới cho ảnh có hồ sơ màu khác, và độ
nhạy ở nơi đó rơi xuống, trong khi không có dòng code nào đổi.

Ba tín hiệu theo dõi, đều lấy ngưỡng từ chương:

* **đầu vào:** chỉ số ổn định phân phối (population stability index, PSI) của độ sáng kênh lục, trên 7
  ngày gần nhất so với 14 ngày đầu của chính phòng khám đó. Chương lấy 0.1 và 0.2 làm mốc, và nói rõ
  đó là quy ước vận hành, không phải một mức ý nghĩa phổ quát;
* **đầu ra:** tỉ lệ chuyển tuyến 7 ngày, báo động khi lệch quá 15 % so với mốc;
* **kết quả:** độ nhạy 7 ngày, báo động khi xuống dưới 88 %. Độ nhạy cần nhãn thật, và ở đây **giả
  định** nhãn của bác sĩ chuyên khoa về sau 14 ngày.

**Dự đoán:** tín hiệu nào báo trước, và trước bao nhiêu ngày?

In [ ]:
BN_NGAY = sach(50, nguon="500 phòng khám · 50 bệnh nhân/ngày")
PSI_CHU_Y, PSI_BAO = sach(0.1, nguon="PSI 0,1 và 0,2"), sach(0.2, nguon="PSI 0,1 và 0,2")
LECH_CHUYEN = sach(15, trich="alert if rate changes more than 15 percent from baseline")
NHAY_BAO = sach(88, trich="seven-day rolling average drops below 88 percent")
LECH_TIN_CAY = sach(10, trich="alert if mean confidence drops more than 10 percent relative to baseline")
SO_NGAY, NGAY_DOI_MAY, TRE_NHAN, CUA_SO, MOC = 98, 28, 14, 7, 14  # giả định của sổ tay
MAY_PK = ["A", "A", "A", "B", "B", "B"]


def sang_kenh_luc(X):
    R = X[:, 0]
    vung = (R > 0.3 * R.amax((1, 2), keepdim=True)).float()
    return ((X[:, 1] * vung).sum((1, 2)) / vung.sum((1, 2))).numpy()


t0 = time.perf_counter()
ngay, pk, nhan, dt_tx, sang, anh_c = [], [], [], [], [], []
for k, may in enumerate(MAY_PK):
    doan = [(may, 0, SO_NGAY)] if k != 1 else [(may, 0, NGAY_DOI_MAY), ("C", NGAY_DOI_MAY, SO_NGAY)]
    for ten_may, d0, d1 in doan:
        X, y = chup_nhieu(BN_NGAY * (d1 - d0), MAY[ten_may], rng)
        ngay.append(np.repeat(np.arange(d0, d1), BN_NGAY))
        pk.append(np.full(len(y), k))
        nhan.append(y)
        dt_tx.append(dac_trung(X, True))
        sang.append(sang_kenh_luc(X))
        if ten_may == "C":
            anh_c = (X, y)
ngay, pk, nhan, sang = map(np.concatenate, (ngay, pk, nhan, sang))
dt_tx = np.concatenate(dt_tx)
ngay_c = np.repeat(np.arange(NGAY_DOI_MAY, SO_NGAY), BN_NGAY)
do_tren_may(f"sinh và tính đặc trưng cho {len(nhan):,} ảnh triển khai", time.perf_counter() - t0, "s")

# Khế ước đầu vào của bước suy luận: đúng kích thước, giá trị trong [0, 1], tìm thấy đĩa võng mạc.
X_c = anh_c[0]
vung_c = (X_c[:, 0] > 0.3 * X_c[:, 0].amax((1, 2), keepdim=True)).float().mean((1, 2))
dat = (X_c.shape[1:] == (3, S, S)) & (X_c.min() >= 0).item() & (X_c.max() <= 1).item() & ((vung_c > 0.35) & (vung_c < 0.7)).numpy()
print(f"ảnh máy C qua khế ước đầu vào: {100 * dat.mean():.1f} %")

Ảnh của máy mới qua mọi kiểm tra ở cửa vào, vì nó vẫn là một ảnh võng mạc hợp lệ. Lỗi không ném ra ở
đâu cả. Giờ chấm điểm mọi ảnh bằng v1, tạm thời cho cả 98 ngày. Phần 6 sẽ thay v1 bằng một phiên bản
mới ở chỗ cần thay.

In [ ]:
diem_v1 = v1.lr.predict_proba(dt_tx)[:, 1]


def theo_doi(diem_tung_anh, nguong_tung_anh):
    """Ba tín hiệu theo ngày cho mỗi phòng khám, và độ nhạy gộp cả mạng lưới."""
    chuyen = diem_tung_anh >= nguong_tung_anh
    tin_cay = np.maximum(diem_tung_anh, 1 - diem_tung_anh)
    kq = {t: np.full((len(MAY_PK), SO_NGAY), np.nan) for t in ("psi", "chuyen", "tin_cay", "nhay")}
    nhay_mang = np.full(SO_NGAY, np.nan)
    for k in range(len(MAY_PK)):
        o = pk == k
        moc = o & (ngay < MOC)
        bien = np.quantile(sang[moc], np.linspace(0, 1, 11))  # 10 ngăn theo thập phân vị của mốc
        bien[0], bien[-1] = -np.inf, np.inf
        p = np.clip(np.histogram(sang[moc], bien)[0] / moc.sum(), 1e-4, None)
        for d in range(SO_NGAY):
            cs = o & (ngay > d - CUA_SO) & (ngay <= d)
            if d >= CUA_SO - 1:
                q = np.clip(np.histogram(sang[cs], bien)[0] / cs.sum(), 1e-4, None)
                kq["psi"][k, d] = np.sum((q - p) * np.log(q / p))
                kq["chuyen"][k, d] = 100 * (chuyen[cs].mean() / chuyen[moc].mean() - 1)
                kq["tin_cay"][k, d] = 100 * (tin_cay[cs].mean() / tin_cay[moc].mean() - 1)
            # nhãn về sau TRE_NHAN ngày: ở ngày d chỉ biết các ca tới ngày d − TRE_NHAN
            cn = o & (ngay > d - TRE_NHAN - CUA_SO) & (ngay <= d - TRE_NHAN) & (nhan == 1)
            if d - TRE_NHAN >= CUA_SO - 1:
                kq["nhay"][k, d] = 100 * chuyen[cn].mean()
    for d in range(TRE_NHAN + CUA_SO - 1, SO_NGAY):
        cn = (ngay > d - TRE_NHAN - CUA_SO) & (ngay <= d - TRE_NHAN) & (nhan == 1)
        nhay_mang[d] = 100 * chuyen[cn].mean()
    return kq, nhay_mang


def ngay_bao_dau(chuoi, dieu_kien, tu_ngay=NGAY_DOI_MAY):
    d = np.where(dieu_kien(chuoi) & (np.arange(SO_NGAY) >= tu_ngay))[0]
    return int(d[0]) if len(d) else None


kq1, mang1 = theo_doi(diem_v1, v1.nguong)
bao = {"PSI > 0.2": ngay_bao_dau(kq1["psi"][1], lambda s: s > PSI_BAO),
       "chuyển tuyến lệch > 15 %": ngay_bao_dau(kq1["chuyen"][1], lambda s: np.abs(s) > LECH_CHUYEN),
       "độ tin cậy giảm > 10 %": ngay_bao_dau(kq1["tin_cay"][1], lambda s: s < -LECH_TIN_CAY),
       "độ nhạy 7 ngày < 88 %": ngay_bao_dau(kq1["nhay"][1], lambda s: s < NHAY_BAO)}
for ten, d in bao.items():
    print(f"PK2, {ten:>26}: " + (f"báo ở ngày {d}, {d - NGAY_DOI_MAY} ngày sau khi đổi máy" if d is not None
                                  else "không báo trong 98 ngày"))
mang_thap = np.nanmin(mang1[NGAY_DOI_MAY + TRE_NHAN:])
do_tren_may("độ nhạy 7 ngày gộp cả mạng lưới, thấp nhất sau khi PK2 đổi máy", mang_thap, "%")
print("mạng lưới gộp " + ("có" if mang_thap < NHAY_BAO else "KHÔNG") + f" xuống dưới {NHAY_BAO} %")

## 6. Quay lại phiên bản trước, hay huấn luyện lại

Khi một cảnh báo bật, chương nhắc rằng ngưỡng trôi chỉ khởi động việc xem xét, không tự chọn cách sửa.
Cách nhanh nhất là quay lại phiên bản trước. Ở ngày 49, nhãn của tuần đầu tiên dùng máy C (ngày 28 tới
34) đã về, đủ để thử điều đó trước khi làm.

**Dự đoán:** v0, mô hình chưa có tiền xử lý, có làm tốt hơn v1 trên ảnh máy C không?

In [ ]:
tuan_dau = ngay_c < NGAY_DOI_MAY + 7
X_tuan, y_tuan = anh_c[0][tuan_dau], anh_c[1][tuan_dau]
for m in (v0, v1):
    nhay, dac_hieu = do_nhay_dac_hieu(m.diem(X_tuan), y_tuan, m.nguong)
    print(f"{m.ten:>28} trên {len(y_tuan)} ảnh máy C: độ nhạy {nhay:.1f} %, độ đặc hiệu {dac_hieu:.1f} %, "
          f"qua cổng: {'có' if qua_cong(nhay, dac_hieu) else 'KHÔNG'}")

Quay lại v0 chỉ đổi một mô hình chưa từng thấy máy C lấy một mô hình khác cũng chưa từng thấy máy C.
Lỗi không nằm ở phiên bản mà ở dữ liệu: đầu vào đã ra khỏi vùng mà mọi phiên bản được huấn luyện.

Vòng phản hồi của chương đưa ra con đường còn lại. Ảnh của PK2 kể từ ngày 28 đang có nhãn dần. Ở ngày
69, nhãn của ngày 28 tới 55 đã về: 1,400 ảnh máy C. Ta chia theo thời gian, ngày 28 tới 41 để huấn luyện,
42 tới 48 để chọn ngưỡng, 49 tới 55 để kiểm tra, rồi cho v2 qua cùng cổng phát hành, ở cả ba loại máy.
Nếu qua, v2 thay v1 ở mọi phòng khám từ ngày 70.

In [ ]:
NGAY_V2 = 70
hl_c, kd_c, kt_c = (ngay_c < 42), (ngay_c >= 42) & (ngay_c < 49), (ngay_c >= 49) & (ngay_c < 56)
C_HL, C_KD, C_KT = [(anh_c[0][m], anh_c[1][m]) for m in (hl_c, kd_c, kt_c)]
v2 = MoHinh("v2: + ảnh máy C có nhãn", torch.cat([LAB_HL[0], B_HL[0], C_HL[0]]),
            np.concatenate([LAB_HL[1], B_HL[1], C_HL[1]]), tien_xu_ly=True).chon_nguong([LAB_KD, B_KD, C_KD])
bang_v2 = bang_cong([v1, v2], {"lab, máy A": LAB_KT, "phòng khám, máy B": B_KT, "PK2, máy C": C_KT})
bang_v2

In [ ]:
diem_v2 = v2.lr.predict_proba(dt_tx)[:, 1]
dung_v2 = ngay >= NGAY_V2
diem_cuoi = np.where(dung_v2, diem_v2, diem_v1)
nguong_cuoi = np.where(dung_v2, v2.nguong, v1.nguong)
kq, mang = theo_doi(diem_cuoi, nguong_cuoi)

ngay_truc = np.arange(SO_NGAY)
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
khac = [k for k in range(len(MAY_PK)) if k != 1]
for k in khac:
    axes[0].semilogy(ngay_truc, kq["psi"][k], color="0.75", lw=1)
    axes[1].plot(ngay_truc, kq["chuyen"][k], color="0.75", lw=1)
    axes[2].plot(ngay_truc, kq["nhay"][k], color="0.75", lw=1)
axes[0].semilogy(ngay_truc, kq["psi"][1], color=MAU[1], lw=2, label="PK2")
axes[0].plot([], [], color="0.75", lw=1, label="năm phòng khám còn lại")
for v, ls, ten in [(PSI_CHU_Y, ":", f"{PSI_CHU_Y}: chú ý"), (PSI_BAO, "--", f"{PSI_BAO}: báo động")]:
    axes[0].axhline(v, color="0.3", ls=ls, lw=1, label=f"PSI {ten} (chương)")
axes[0].set_ylabel("PSI độ sáng kênh lục, 7 ngày")
axes[0].set_title("Đầu vào: PSI so với 14 ngày đầu của phòng khám")

axes[1].fill_between(ngay_truc, -LECH_CHUYEN, LECH_CHUYEN, color=MAU[2], alpha=0.12, lw=0,
                     label=f"±{LECH_CHUYEN} % quanh mốc (chương)")
axes[1].plot(ngay_truc, kq["chuyen"][1], color=MAU[1], lw=2, label="PK2")
axes[1].set_ylabel("tỉ lệ chuyển tuyến 7 ngày,\nlệch so với mốc (%)")
axes[1].set_title("Đầu ra: tỉ lệ chuyển tuyến, không cần nhãn")

axes[2].plot(ngay_truc, kq["nhay"][1], color=MAU[1], lw=2, label="PK2")
axes[2].plot(ngay_truc, mang, color=MAU[0], lw=2, label="gộp cả sáu phòng khám")
axes[2].axhline(NHAY_BAO, color="0.3", ls="--", lw=1, label=f"báo động {NHAY_BAO} % (chương)")
axes[2].set_ylabel("độ nhạy 7 ngày (%)\ntheo ngày nhãn về")
axes[2].set_title(f"Kết quả: độ nhạy, khi nhãn về sau {TRE_NHAN} ngày")
axes[2].set_xlabel("ngày kể từ khi triển khai v1")
for ax in axes:
    ax.axvline(NGAY_DOI_MAY, color=MAU[1], lw=1, alpha=0.6)
    ax.axvline(NGAY_V2, color=MAU[2], lw=1, alpha=0.8)
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8)
axes[0].set_ylim(1e-3, 30)
axes[0].text(NGAY_DOI_MAY + 0.8, 1.6e-3, "PK2 đổi sang máy C", fontsize=8, color=MAU[1])
axes[0].text(NGAY_V2 + 0.8, 1.6e-3, "triển khai v2", fontsize=8, color=MAU[2])
axes[2].set_xlim(0, SO_NGAY - 1)
fig.suptitle("Theo dõi sáu phòng khám trong 14 tuần (ảnh giả lập)", fontsize=11)
plt.tight_layout()
plt.show()
d_hoi_phuc = ngay_bao_dau(kq["nhay"][1], lambda s: s >= NHAY_BAO, tu_ngay=NGAY_V2)
print(f"độ nhạy của PK2 về lại trên {NHAY_BAO} % "
      + (f"ở ngày {d_hoi_phuc}, {d_hoi_phuc - NGAY_V2} ngày sau khi triển khai v2" if d_hoi_phuc is not None
         else "chưa xảy ra trong 98 ngày"))

Ba điều đọc được từ đồ thị và các dòng in phía trên:

* **Tín hiệu không cần nhãn đến trước.** PSI và tỉ lệ chuyển tuyến chỉ cần ảnh và điểm số, nên phản
  ứng trong vài ngày. Độ nhạy là thứ ta thật sự quan tâm, nhưng nó chỉ biết được khi nhãn về, ở đây
  chậm ít nhất 14 ngày. Độ tin cậy trung bình là một tín hiệu không cần nhãn khác, nhưng nó chỉ báo khi
  mô hình *kém chắc chắn hơn*. Một mô hình bỏ sót bệnh mà vẫn tự tin thì không làm nó giảm.
* **Con số gộp che mất một phòng khám.** PK2 chỉ là một trong sáu nơi, nên độ nhạy gộp của mạng lưới
  chỉ nhích xuống. Theo dõi phải tách theo phòng khám, cũng như đánh giá phải tách theo nhóm.
* **Sau khi sửa, mọi mốc so sánh phải đặt lại.** PSI của PK2 vẫn cao, vì PSI đo việc đầu vào đổi, không
  đo việc mô hình sai, và đầu vào vẫn là máy C. Tỉ lệ chuyển tuyến của các phòng khám khác cũng đổi khi
  v2 thay v1, dù máy chụp của họ không đổi: một mô hình mới có điểm vận hành mới. Nếu không đặt lại mốc,
  cảnh báo sẽ bật mãi, và người vận hành sẽ học cách bỏ qua nó.

Vòng lặp vừa chạy là vòng phản hồi (feedback loop) của chương: theo dõi phát hiện, nhãn từ thực địa
thành dữ liệu huấn luyện, và mô hình mới lại qua cùng cổng kiểm định trước khi triển khai. Nó cũng là
lý do chương gọi việc coi chuẩn bị dữ liệu là bước làm một lần là một cạm bẫy: tiền xử lý của v1 gỡ được
độ sáng và cân bằng trắng của máy B, nhưng không gỡ được đường cong độ sáng của máy C.

## Thử thêm

1. Ở phần 2, đổi `SEED` khi tạo `rng` (ví dụ `np.random.default_rng(3)`) và chạy lại cả sổ tay. v0 còn
   trượt cổng ở máy B không? Một kết luận kiểm định dựa trên một tập dữ liệu cũng là một phép đo có
   nhiễu.
2. Ở phần 5, đổi `TRE_NHAN` thành 30 ngày. Cảnh báo độ nhạy đến muộn bao nhiêu, và v2 có còn được
   huấn luyện kịp ở ngày 70 không?
3. Đổi `gamma` của `MAY["C"]` thành 1.0 và `can_bang` thành `(0.9, 1.2, 1.3)`: máy mới chỉ đổi cân
   bằng trắng, không đổi đường cong độ sáng. PSI có còn báo không? Độ nhạy của v1 ở PK2 có còn rơi
   không, và vì sao? Một cảnh báo ở đầu vào chưa chắc là một lỗi ở kết quả.

## Tóm lại

* Bước định nghĩa bài toán biến yêu cầu lâm sàng thành hai cổng, độ nhạy trên 90 % và độ đặc hiệu
  trên 80 %, và mọi bước sau đều được đo bằng chúng.
* Qua cổng ở lab chưa phải là sẵn sàng triển khai. Kiểm định ở nhiều nơi và tách theo nhóm cho thấy
  điều mà một con số gộp che mất.
* Khi lỗi đến từ cách dữ liệu được chụp, thêm dữ liệu cùng loại không chữa được; tiền xử lý và dữ liệu
  thực địa thì chữa được.
* Sau khi triển khai, một máy chụp mới làm mô hình suy giảm âm thầm mà không có dòng code nào đổi. Tín
  hiệu đầu vào và đầu ra báo trước tín hiệu cần nhãn, và phải được theo dõi theo từng phòng khám.
* Quay lại phiên bản trước không chữa được một dịch chuyển phân phối (distribution shift). Huấn luyện
  lại bằng nhãn từ vòng phản hồi, rồi qua lại cổng phát hành, thì chữa được.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Quy trình hệ thống học máy](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch03-ml-workflow/), dựng từ chương
[*ML Workflow*](https://mlsysbook.ai/vol1/ml_workflow/ml_workflow.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.